## Setup
- download [dataset](https://www.kaggle.com/competitions/dfl-bundesliga-data-shootout/data) using [Kaggle API](https://github.com/Kaggle/kaggle-api), 150 videos that we can use 
for training
- [player dataset](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc)
- pip install ultralytics to get access to YOLO https://docs.ultralytics.com/quickstart/

## Libaries

In [1]:
import cv2
from ultralytics import YOLO

## Load Model

In [2]:
#define the YOLO model
model = YOLO('yolov8x')

## Object Detection

### Inference on video clip
The video file is passed to the model frame by frame where objects are detected, boundary boxes are drawn and an output video file is created with the overlay.

### Detected Objects

YOLOv8 is capable of detecting 80 different objects in images and video clips. For this task I am just looking for two classes, a `Person` and a `Sports Ball`. The full list of classes however can be seen below:

| Index |     Object     | Index |     Object     | Index |     Object     | Index |     Object     |
|:-----:|:--------------:|:-----:|:--------------:|:-----:|:--------------:|:-----:|:--------------:|
|   **0**   |    **person**      |  20   |    elephant    |  40   |   wine glass   |  60   |  dining table  |
|   1   |    bicycle     |  21   |     bear       |  41   |      cup       |  61   |     toilet     |
|   2   |      car       |  22   |     zebra      |  42   |      fork      |  62   |       tv       |
|   3   |  motorcycle   |  23   |    giraffe     |  43   |     knife      |  63   |     laptop     |
|   4   |   airplane     |  24   |   backpack     |  44   |     spoon      |  64   |     mouse      |
|   5   |      bus       |  25   |    umbrella    |  45   |      bowl      |  65   |     remote     |
|   6   |     train      |  26   |    handbag     |  46   |    banana      |  66   |   keyboard     |
|   7   |     truck      |  27   |      tie       |  47   |     apple      |  67   |  cell phone    |
|   8   |     boat       |  28   |    suitcase    |  48   |   sandwich     |  68   |   microwave    |
|   9   | traffic light  |  29   |    frisbee     |  49   |    orange      |  69   |     oven       |
|  10   | fire hydrant   |  30   |     skis       |  50   |   broccoli     |  70   |    toaster     |
|  11   |   stop sign    |  31   |   snowboard    |  51   |    carrot      |  71   |      sink      |
|  12   | parking meter  |  **32**   |  **sports ball**   |  52   |    hot dog     |  72   | refrigerator   |
|  13   |     bench      |  33   |     kite       |  53   |     pizza      |  73   |     book       |
|  14   |      bird      |  34   | baseball bat   |  54   |     donut      |  74   |     clock      |
|  15   |      cat       |  35   | baseball glove |  55   |     cake       |  75   |     vase       |
|  16   |      dog       |  36   |   skateboard   |  56   |     chair      |  76   |    scissors    |
|  17   |     horse      |  37   |    surfboard   |  57   |     couch      |  77   |  teddy bear    |
|  18   |     sheep      |  38   |  tennis racket |  58   | potted plant   |  78   |   hair drier   |
|  19   |      cow       |  39   |     bottle     |  59   |      bed       |  79   |   toothbrush   |


We can loop through the results to see what was detected. These results, provide comprehensive information about detected objects, their locations, classes, and confidence scores. `cls: tensor([0.])` is a `Person`, `cls: tensor([32.])` is the sports ball. Approximately 20 people were found in the video clip and 1 ball. However ball tracking is not very good, the model is detecting both players on pitch and spectators off the pitch. Also we are not using colours to distingish teams or the referee.

![Football Tracking Output](https://raw.githubusercontent.com/rob-sullivan/ai/football-tracking/football-tracking/output.PNG)


### Improved Object Tracking
By using [ByteTrack](https://github.com/ifzhang/ByteTrack.git) we can improve [multi-object tracking](https://github.com/ultralytics/ultralytics/tree/main/ultralytics/trackers), since it employs a novel technique to instead ignoring low threhold objects, track their similar trajectories across frames.

The following was done to work with ByteTrack:
* https://www.youtube.com/watch?v=6LGpf-a1K1Q
* https://youtu.be/LNwODJXcvt4
* Track using ByteTrack tracker
```shell
    > yolo track model=path/to/best.pt tracker="bytetrack.yaml"
```

## Object Tracking
ref: https://www.youtube.com/watch?v=uMzOcCNKr5A&t=644s

### Load Video

In [5]:
video_path = './data/08fd33_4.mp4'
cap = cv2.VideoCapture(video_path)

# Define the codec and create a VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Using 'XVID' codec for MP4 files
fps = cap.get(cv2.CAP_PROP_FPS)  # Get the FPS of the input video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('./data/08fd33_4_tracked.mp4', fourcc, fps, (width, height))

### Read Frames

In [9]:
# Initialize detection model
initial_detection = True
tracker = None

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model.track(frame, persist=True, tracker="bytetrack.yaml")
    
    # Plot results
    frame_ = results[0].plot()
    
    # Resize the image if needed
    width = int(frame_.shape[1] * 0.5)  # Resize to 50% of the original width
    height = int(frame_.shape[0] * 0.5) # Resize to 50% of the original height
    dim = (width, height)
    
    resized_frame_ = cv2.resize(frame_, dim, interpolation=cv2.INTER_AREA)

    # Write the frame to the video file
    out.write(resized_frame_)

    # Optional: Display the frame (remove if you only want to save the video)
    # cv2.imshow('Football Tracker', resized_frame_)
    # if cv2.waitKey(25) & 0xFF == ord('q'):
    #     break

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()


0: 384x640 22 persons, 708.0ms
Speed: 3.0ms preprocess, 708.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 22 persons, 683.0ms
Speed: 3.0ms preprocess, 683.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 23 persons, 728.9ms
Speed: 3.0ms preprocess, 728.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 22 persons, 684.0ms
Speed: 2.0ms preprocess, 684.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 22 persons, 679.0ms
Speed: 3.0ms preprocess, 679.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 22 persons, 722.9ms
Speed: 4.0ms preprocess, 722.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 22 persons, 687.0ms
Speed: 4.0ms preprocess, 687.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 22 persons, 687.0ms
Speed: 2.0ms preprocess, 687.0ms inference, 1.0ms postproc